# UKE Patient Goals — ICF Annotation Results

Automatic ICF annotation of 529 UKE patient goal concepts using **llama4:latest** (best model from Phase 1).

## Annotation setup

| | Detail |
|---|---|
| **Model** | llama4:latest (108.6B MoE, best from 4-cohort evaluation) |
| **Input** | 529 German patient goal concepts (`set_goal`) from UKE |
| **Output** | ICF component (b/s/d/e) + chapter (b1-b8, s1-s8, d1-d9, e1-e5) + confidence |
| **Conditions** | Zero-shot (no examples) vs RAG (top-6 retrieved 4-cohort examples) |

## ICF classification levels predicted
- **Component:** b (Body Functions) · s (Body Structures) · d (Activities & Participation) · e (Environmental Factors)
- **Chapter:** e.g. d4 = Mobility, d5 = Self-care, b7 = Neuromusculoskeletal functions ...

---
*Reference: Cieza et al. 2005 ICF Linking Rules | BfArM ICF 2005 classification*

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

REPO   = Path("__file__").resolve().parent.parent
OUTPUT = REPO / "output"
sys.path.insert(0, str(REPO / "scripts"))

from utils.data_utils import load_splits

sns.set_theme(style="whitegrid", font_scale=1.1)

# ── Chapter labels ──────────────────────────────────────────────────────────
CHAPTER_LABELS = {
    "b1": "b1 Mental", "b2": "b2 Sensory/Pain", "b3": "b3 Voice/Speech",
    "b4": "b4 Cardiovascular", "b5": "b5 Digestive", "b6": "b6 Urogenital",
    "b7": "b7 Neuromusculoskeletal", "b8": "b8 Skin",
    "s1": "s1 Nervous", "s2": "s2 Eye/Ear", "s3": "s3 Voice",
    "s4": "s4 Cardiovascular", "s5": "s5 Digestive", "s6": "s6 Urogenital",
    "s7": "s7 Movement", "s8": "s8 Skin",
    "d1": "d1 Learning", "d2": "d2 General Tasks", "d3": "d3 Communication",
    "d4": "d4 Mobility", "d5": "d5 Self-care", "d6": "d6 Domestic Life",
    "d7": "d7 Interpersonal", "d8": "d8 Life Areas", "d9": "d9 Community",
    "e1": "e1 Products", "e2": "e2 Environment", "e3": "e3 Support",
    "e4": "e4 Attitudes", "e5": "e5 Services",
}

COMP_LABELS = {
    "b": "Body Functions (b)",
    "s": "Body Structures (s)",
    "d": "Activities & Participation (d)",
    "e": "Environmental Factors (e)",
}

print("Setup complete.")

---
## 1. Load Data

In [ ]:
# Load UKE annotations
ann_path = OUTPUT / "uke_annotations.csv"
df = pd.read_csv(ann_path)

# Load 4-cohort training data for comparison
train_df, _ = load_splits()

print(f"UKE concepts: {len(df)}")
print(f"4-cohort training concepts: {len(train_df)}")
print(f"\nUKE columns: {list(df.columns)}")
print(f"\nSample UKE data:")
display(df.head(10))

---
## 2. Data Quality

In [ ]:
n = len(df)

# Valid predictions
valid_comp_z = (df["comp_zeroshot"] != "?").sum()
valid_comp_r = (df["comp_rag"] != "?").sum()
valid_chap_z = (df["chapter_zeroshot"] != "?").sum()
valid_chap_r = (df["chapter_rag"] != "?").sum()

quality = pd.DataFrame({
    "Condition": ["Zero-shot", "RAG"],
    "Valid component": [f"{valid_comp_z}/{n} ({valid_comp_z/n*100:.1f}%)",
                        f"{valid_comp_r}/{n} ({valid_comp_r/n*100:.1f}%)"],
    "Valid chapter":   [f"{valid_chap_z}/{n} ({valid_chap_z/n*100:.1f}%)",
                        f"{valid_chap_r}/{n} ({valid_chap_r/n*100:.1f}%)"],
}).set_index("Condition")

display(quality.style.set_caption("Annotation quality — valid predictions out of 529"))

# Confidence distribution
conf_df = pd.DataFrame({
    "Zero-shot": df["conf_zeroshot"].value_counts(),
    "RAG":       df["conf_rag"].value_counts(),
}).reindex(["high", "medium", "low"]).fillna(0).astype(int)
conf_df["Zero-shot %"] = (conf_df["Zero-shot"] / n * 100).round(1)
conf_df["RAG %"]       = (conf_df["RAG"]       / n * 100).round(1)

display(conf_df.style.background_gradient(cmap="YlGn", subset=["Zero-shot %", "RAG %"])
               .set_caption("Confidence distribution"))

---
## 3. Agreement: Zero-shot vs RAG

In [ ]:
agree_comp = (df["comp_zeroshot"] == df["comp_rag"]).mean() * 100
agree_chap = (df["chapter_zeroshot"] == df["chapter_rag"]).mean() * 100

print(f"Agreement between Zero-shot and RAG:")
print(f"  Component level: {agree_comp:.1f}%")
print(f"  Chapter level:   {agree_chap:.1f}%")

# Disagreement examples
disagree = df[df["comp_zeroshot"] != df["comp_rag"]]
print(f"\nConcepts where component disagrees: {len(disagree)}")
if len(disagree) > 0:
    display(
        disagree[["set_goal", "comp_zeroshot", "chapter_zeroshot",
                  "comp_rag", "chapter_rag"]].head(10)
          .style.set_caption("Sample disagreements between zero-shot and RAG")
    )

---
## 4. Component Prevalence — UKE vs 4-Cohort

In [ ]:
# 4-cohort component distribution
train_comp = train_df["primary_code"].str[0].str.lower()
train_comp = train_comp[train_comp.isin(["b","s","d","e"])]

datasets = {
    "UKE (Zero-shot)": df["comp_zeroshot"],
    "UKE (RAG)":       df["comp_rag"],
    "4-cohort train":  train_comp,
}

# Summary table
comp_table = pd.DataFrame({
    label: ser.value_counts(normalize=True).mul(100).round(1)
    for label, ser in datasets.items()
}).reindex(["d","b","e","s"]).fillna(0)
comp_table.index = [COMP_LABELS.get(c, c) for c in comp_table.index]

display(
    comp_table.style
      .format("{:.1f}%")
      .background_gradient(cmap="Blues", axis=None)
      .set_caption("ICF Component Distribution (%) — UKE vs 4-Cohort training data")
)

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
colors = ["#4C72B0", "#55A868", "#DD8452"]

for ax, (title, ser), color in zip(axes, datasets.items(), colors):
    counts = ser.value_counts(normalize=True).mul(100).reindex(["d","b","e","s"], fill_value=0)
    labels = [COMP_LABELS[c] for c in counts.index]
    bars = ax.bar(labels, counts.values, color=color, edgecolor="white", width=0.6)
    for bar, v in zip(bars, counts.values):
        if v > 0.5:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f"{v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_ylabel("% of concepts")
    ax.set_ylim(0, 105)
    ax.tick_params(axis="x", rotation=25)

fig.suptitle("ICF Component Prevalence: Which domains do rehabilitation goals target?",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT / "fig_uke_component_prevalence.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → output/fig_uke_component_prevalence.png")

---
## 5. Chapter Prevalence (RAG)

In [ ]:
VALID_CHAPTERS = set(CHAPTER_LABELS.keys())

# Chapter counts for UKE (RAG) — only valid predictions
uke_chap = df[df["chapter_rag"] != "?"]["chapter_rag"]
chap_counts = uke_chap.value_counts()

# 4-cohort chapters — filter to valid ICF chapters only (exclude nd/nc/pf/hc)
train_chap_all   = train_df["primary_code"].str[:2].str.lower()
train_chap_valid = train_chap_all[train_chap_all.isin(VALID_CHAPTERS)]
train_chap_counts = train_chap_valid.value_counts()

# Summary table
chap_table = pd.DataFrame({
    "UKE (RAG)":      chap_counts,
    "4-cohort train": train_chap_counts,
}).fillna(0).astype(int)
chap_table.index = [CHAPTER_LABELS.get(c, c) for c in chap_table.index]
chap_table = chap_table.sort_values("UKE (RAG)", ascending=False)
chap_table["UKE %"]       = (chap_table["UKE (RAG)"]      / len(uke_chap)          * 100).round(1)
chap_table["4-cohort %"]  = (chap_table["4-cohort train"]  / len(train_chap_valid)  * 100).round(1)

display(
    chap_table.style
      .format("{:.0f}", subset=["UKE (RAG)", "4-cohort train"])
      .format("{:.1f}%", subset=["UKE %", "4-cohort %"])
      .background_gradient(cmap="YlGn", subset=["UKE %"])
      .set_caption("ICF Chapter Distribution — UKE vs 4-Cohort")
)

# Bar chart — top 12 chapters
top_chap = chap_counts.head(12)
top_labels = [CHAPTER_LABELS.get(c, c) for c in top_chap.index]

fig, ax = plt.subplots(figsize=(13, 5))
colors_bar = ["#4C72B0" if c.startswith("d") else
              "#DD8452" if c.startswith("b") else
              "#55A868" if c.startswith("e") else "#C44E52"
              for c in top_chap.index]
bars = ax.bar(top_labels, top_chap.values, color=colors_bar, edgecolor="white")
for bar, v in zip(bars, top_chap.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{v}\n({v/len(uke_chap)*100:.1f}%)",
            ha="center", va="bottom", fontsize=8.5)
ax.set_ylabel("Number of UKE concepts")
ax.set_title("Top ICF Chapters in UKE Goals (RAG annotation)", fontweight="bold")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig(OUTPUT / "fig_uke_chapter_prevalence.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → output/fig_uke_chapter_prevalence.png")

---
## 6. UKE vs 4-Cohort: Side-by-side Comparison

In [ ]:
# All chapters present in either dataset
all_chapters = sorted(set(list(uke_chap.unique()) + list(train_chap_counts.index)))

uke_pct   = uke_chap.value_counts().reindex(all_chapters, fill_value=0) / len(uke_chap) * 100
train_pct = train_chap_counts.reindex(all_chapters, fill_value=0) / len(train_chap_valid) * 100

# Only show chapters with at least 1% in either dataset
mask = (uke_pct >= 1) | (train_pct >= 1)
uke_pct_f   = uke_pct[mask]
train_pct_f = train_pct[mask]
labels_f    = [CHAPTER_LABELS.get(c, c) for c in uke_pct_f.index]

x = np.arange(len(labels_f))
bar_w = 0.38

fig, ax = plt.subplots(figsize=(15, 5))
bars1 = ax.bar(x - bar_w/2, uke_pct_f.values,  bar_w, label="UKE (RAG)",      color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + bar_w/2, train_pct_f.values, bar_w, label="4-cohort train", color="#DD8452", edgecolor="white")

for bars in [bars1, bars2]:
    for bar in bars:
        v = bar.get_height()
        if v > 0.5:
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.3,
                    f"{v:.1f}%", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(labels_f, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("% of concepts")
ax.set_title("ICF Chapter Distribution: UKE goals vs 4-Cohort training data",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "fig_uke_vs_4cohort_chapters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → output/fig_uke_vs_4cohort_chapters.png")

---
## 7. Sample Annotations

In [ ]:
# Sample by chapter to show variety
print("=== Sample annotations per chapter (RAG) ===")
for chap in ["d4", "d5", "b7", "b4", "d2", "e1"]:
    subset = df[df["chapter_rag"] == chap][["set_goal", "comp_rag", "chapter_rag", "conf_rag"]]
    if len(subset) == 0:
        continue
    sample = subset.sample(min(3, len(subset)), random_state=42)
    label = CHAPTER_LABELS.get(chap, chap)
    sample.columns = ["Konzept", "Component", "Chapter", "Confidence"]
    display(sample.reset_index(drop=True).style.set_caption(f"{label} — {len(subset)} concepts"))
    print()

---
## 8. High-confidence vs Low-confidence Predictions

In [ ]:
# Where is the model uncertain?
low_conf = df[df["conf_rag"].isin(["low", "medium"])].copy()
print(f"Low/medium confidence predictions: {len(low_conf)} ({len(low_conf)/len(df)*100:.1f}%)")
print(f"High confidence predictions:       {(df['conf_rag']=='high').sum()} ({(df['conf_rag']=='high').mean()*100:.1f}%)")

# Show uncertain cases
if len(low_conf) > 0:
    display(
        low_conf[["set_goal", "comp_zeroshot", "chapter_zeroshot",
                  "comp_rag", "chapter_rag", "conf_rag"]]
          .head(15)
          .reset_index(drop=True)
          .style.set_caption("Low/medium confidence predictions — consider manual review")
    )

# Also show where zero-shot and RAG disagree
disagree_chap = df[df["chapter_zeroshot"] != df["chapter_rag"]]
print(f"\nChapter disagreements: {len(disagree_chap)} ({len(disagree_chap)/len(df)*100:.1f}%)")
if len(disagree_chap) > 0:
    display(
        disagree_chap[["set_goal", "comp_zeroshot", "chapter_zeroshot",
                       "comp_rag", "chapter_rag"]]
          .head(10)
          .reset_index(drop=True)
          .style.set_caption("Concepts where zero-shot and RAG disagree on chapter")
    )

---
## 9. Summary

In [ ]:
print("=" * 60)
print("UKE ANNOTATION SUMMARY")
print("=" * 60)
print(f"Total UKE concepts annotated: {len(df)}")
print(f"Model used: llama4:latest (108.6B MoE)")

print(f"\n--- Data Quality ---")
print(f"  Valid component (RAG): {(df['comp_rag'] != '?').mean()*100:.1f}%")
print(f"  Valid chapter (RAG):   {(df['chapter_rag'] != '?').mean()*100:.1f}%")
print(f"  High confidence (RAG): {(df['conf_rag']=='high').mean()*100:.1f}%")

print(f"\n--- Zero-shot vs RAG Agreement ---")
print(f"  Component level: {(df['comp_zeroshot']==df['comp_rag']).mean()*100:.1f}%")
print(f"  Chapter level:   {(df['chapter_zeroshot']==df['chapter_rag']).mean()*100:.1f}%")

print(f"\n--- Component Distribution (RAG) ---")
for comp in ["d", "b", "e", "s"]:
    count = (df["comp_rag"] == comp).sum()
    pct = count / len(df) * 100
    print(f"  {COMP_LABELS[comp]}: {count} ({pct:.1f}%)")

print(f"\n--- Top 5 Chapters (RAG) ---")
for chap, count in df["chapter_rag"].value_counts().head(5).items():
    label = CHAPTER_LABELS.get(chap, chap)
    print(f"  {label}: {count} ({count/len(df)*100:.1f}%)")

print(f"\n--- Key Finding ---")
d_pct = (df["comp_rag"] == "d").mean() * 100
d4_pct = (df["chapter_rag"] == "d4").mean() * 100
print(f"  {d_pct:.1f}% of UKE goals are Activity/Participation (d)")
print(f"  {d4_pct:.1f}% are specifically Mobility (d4)")
print(f"  → UKE rehabilitation goals are predominantly focused on mobility and self-care")

print(f"\nOutput files:")
for f in sorted(OUTPUT.glob("uke*")):
    print(f"  {f.name:<45} {f.stat().st_size/1024:.0f} KB")
print("=" * 60)